# ROS Model Training Pipeline

**Purpose:** Train Rest of Season (ROS) prediction models from historical data

**Last Updated:** 2025-10-08

---

## Overview
This notebook trains the complete ROS ensemble system:
- **DirectROSForecaster** (50% weight): Multi-quantile model with time series lags + all features
- **DartsTemporalEnsemble** (40% weight): TCN + TSMixer + AutoARIMA on pure WAR trajectories
- **Baseline MultiQuantileHistGB** (10% weight): Pure feature-based quantile regression

## Training Data Structure (Multipoint Splits)
Historical full seasons (2016-2024) are split into **multiple** training samples per season:
- **25% split**: Stats through ~40 games → predict remaining 122 games WAR
- **50% split**: Stats through ~81 games → predict remaining 81 games WAR
- **75% split**: Stats through ~121 games → predict remaining 41 games WAR

This creates 3x more training data and teaches the model about **season timing** (early-season SSS vs late-season).

## Key Innovation: Flexible Timing
The `season_completion_pct` feature allows projections at **any point** in the season:
- Not limited to fixed firsthalf/secondhalf splits
- Works for 3 weeks into season (9%), All-Star break (50%), or any custom point
- Model learns timing effects from multiple split points and interpolates

In [1]:
# Cell 1: Imports and Setup

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
from datetime import datetime

# Add project root to path
project_root = Path('.').absolute().parent.parent
sys.path.insert(0, str(project_root))

from new_pipeline.models.ros import (
    HitterROSEnsemble,
    PitcherROSEnsemble,
    ROS_HITTER_FEATURES,
    ROS_PITCHER_FEATURES,
    prepare_ros_training_data,
    temporal_cv_split,
    calculate_ros_metrics
)
from new_pipeline.common.features import ROSFeatureBuilder
from new_pipeline.common.data_preparation import create_multipoint_splits
from new_pipeline.notebooks.shared.pipeline_runner import load_historical_data

print("Imports successful!")
print(f"ROS Hitter Features: {len(ROS_HITTER_FEATURES)}")
print(f"ROS Pitcher Features: {len(ROS_PITCHER_FEATURES)}")

16:30:30 - new_pipeline.common.logging_config - INFO - Logging module initialized for new_pipeline
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\fs\__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports successful!
ROS Hitter Features: 65
ROS Pitcher Features: 51


In [2]:
# Cell 2: Load and Process Historical Data (2016-2024)

print("="*70)
print("LOADING HISTORICAL FULL-SEASON DATA (2016-2024)")
print("="*70)

from new_pipeline.notebooks.shared.pipeline_runner import run_data_pipeline

# Load raw historical data
print("\nLoading raw hitter data...")
hitter_raw = load_historical_data(
    player_type='hitter',
    years=range(2016, 2025)  # 2016-2024
)

print("\nLoading raw pitcher data...")
pitcher_raw = load_historical_data(
    player_type='pitcher',
    years=range(2016, 2025)  # 2016-2024
)

print(f"\nLoaded {len(hitter_raw)} hitter seasons (raw)")
print(f"Loaded {len(pitcher_raw)} pitcher seasons (raw)")

# Process through pipeline (adds features + Age + WAR_per_600/WAR_per_162)
print("\nProcessing hitter data through pipeline...")
print("  Pipeline adds: Age (from BP_Data), features, WAR_per_600")
hitter_processed = run_data_pipeline(hitter_raw, 'hitter')

print("\nProcessing pitcher data through pipeline...")
print("  Pipeline adds: Age (from BP_Data), features, WAR_per_162")
pitcher_processed = run_data_pipeline(pitcher_raw, 'pitcher')

print(f"\nProcessed {len(hitter_processed)} qualified hitters")
print(f"Processed {len(pitcher_processed)} qualified pitchers")
print(f"Years: {sorted(hitter_processed['Year'].unique())}")

# Verify Age and WAR rates are present
print("\nData quality checks:")
print(f"  Hitters with Age: {hitter_processed['Age'].notna().sum()}")
print(f"  Hitters with WAR_per_600: {hitter_processed['WAR_per_600'].notna().sum()}")
print(f"  Pitchers with Age: {pitcher_processed['Age'].notna().sum()}")
print(f"  Pitchers with WAR_per_162: {pitcher_processed['WAR_per_162'].notna().sum()}")

if 'Age' in hitter_processed.columns:
    print(f"\n  Hitter Age range: [{hitter_processed['Age'].min():.0f}, {hitter_processed['Age'].max():.0f}]")
    print(f"  Pitcher Age range: [{pitcher_processed['Age'].min():.0f}, {pitcher_processed['Age'].max():.0f}]")

LOADING HISTORICAL FULL-SEASON DATA (2016-2024)

Loading raw hitter data...

Loading raw pitcher data...

Loaded 5760 hitter seasons (raw)
Loaded 7237 pitcher seasons (raw)

Processing hitter data through pipeline...
  Pipeline adds: Age (from BP_Data), features, WAR_per_600


16:30:39 - new_pipeline.common.transformers.filters - INFO - PAFilter: Removed 1524 hitters with < 75 PA (full season)
16:30:39 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Loaded Age for 2088 hitters
16:30:39 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Added Age column (range: 20-45)
16:30:39 - new_pipeline.common.transformers.hitter_features - INFO - Loading hitter features...
16:30:50 - new_pipeline.common.transformers.hitter_features - INFO - Loaded 11 hitter feature sets (33 total columns)
16:30:50 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Learned replacement values for 28 features
16:30:50 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Imputed 70 missing values
16:30:50 - new_pipeline.common.transformers.validators - WARNING - FeatureValidator found issues:
  - Feature 'K%' range [3.09, 51.25] outside expected [0, 50]
  - Feature 'AVG' range [0.09, 0.38] outside expec


Processing pitcher data through pipeline...
  Pipeline adds: Age (from BP_Data), features, WAR_per_162


16:30:50 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Loaded Age for 2272 pitchers
16:30:50 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Added Age column (range: 19-45)
16:30:50 - new_pipeline.common.transformers.pitcher_features - INFO - Loading pitcher features...


16:30:55 - new_pipeline.common.transformers.pitcher_features - INFO - Loaded 13 pitcher feature sets (38 total columns)
16:30:55 - new_pipeline.common.transformers.pitcher_composite_transformer - INFO - Calculating pitcher composite features...
16:30:55 - new_pipeline.common.transformers.pitcher_composite_transformer - INFO - Added 7 composite features
16:30:55 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Learned replacement values for 41 features
16:30:56 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Imputed 10901 missing values
16:30:56 - new_pipeline.common.transformers.validators - WARNING - FeatureValidator found issues:
  - Feature 'BB%' range [0.00, 50.00] outside expected [0, 25]
  - Feature 'K%' range [0.00, 53.00] outside expected [0, 50]
  - Feature 'ERA' range [0.00, 37.50] outside expected [0, 15]
  - Feature 'GB%' range [0.00, 82.79] outside expected [20, 80]
16:30:56 - new_pipeline.common.transformers.feature_s


Processed 4236 qualified hitters
Processed 5652 qualified pitchers
Years: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Data quality checks:
  Hitters with Age: 4236
  Hitters with WAR_per_600: 4236
  Pitchers with Age: 5652
  Pitchers with WAR_per_162: 5652

  Hitter Age range: [20, 45]
  Pitcher Age range: [19, 45]


In [3]:
# Cell 3: Create Multipoint Season Splits

print("="*70)
print("CREATING MULTIPOINT SEASON SPLITS")
print("="*70)

# Create splits at 25%, 50%, 75% of season
# Use PROCESSED data (has Age, WAR_per_600, all features)
split_points = [0.25, 0.5, 0.75]

print(f"\nCreating splits at: {split_points}")
print("Each player-season becomes 3 training samples")
print("  - 25% split: 'At 40 games, predict remaining 122 games WAR'")
print("  - 50% split: 'At 81 games, predict remaining 81 games WAR'")
print("  - 75% split: 'At 121 games, predict remaining 41 games WAR'")

print("\nSplitting hitter data (from PROCESSED - has Age + WAR_per_600)...")
hitter_splits = create_multipoint_splits(
    full_season_df=hitter_processed,  # Use processed data
    split_points=split_points,
    player_type='hitter',
    season_length=162
)

print("\nSplitting pitcher data (from PROCESSED - has Age + WAR_per_162)...")
pitcher_splits = create_multipoint_splits(
    full_season_df=pitcher_processed,  # Use processed data
    split_points=split_points,
    player_type='pitcher',
    season_length=162
)

print(f"\nHitter splits: {len(hitter_splits)} samples (from {len(hitter_processed)} full seasons)")
print(f"Pitcher splits: {len(pitcher_splits)} samples (from {len(pitcher_processed)} full seasons)")

# Verify split structure and Age preservation
print("\nSample split row (hitter):")
sample = hitter_splits.iloc[0]
print(f"  Player: {sample.get('Name', 'N/A')}")
print(f"  Year: {sample['Year']}")
print(f"  Age: {sample.get('Age', 'MISSING')}")
print(f"  Split point: {sample['split_point']}")
print(f"  Season completion: {sample['season_completion_pct']:.1%}")
print(f"  Games played: {sample['games_played']:.0f}")
print(f"  Current PA: {sample['current_PA']:.0f}")
print(f"  Remaining PA: {sample['remaining_PA']:.0f}")
print(f"  Remaining WAR (target): {sample['remaining_WAR']:.2f}")

# Verify Age is preserved
print(f"\nAge preserved in splits:")
print(f"  Hitters: {'Age' in hitter_splits.columns} (range: {hitter_splits['Age'].min():.0f}-{hitter_splits['Age'].max():.0f})")
print(f"  Pitchers: {'Age' in pitcher_splits.columns} (range: {pitcher_splits['Age'].min():.0f}-{pitcher_splits['Age'].max():.0f})")

CREATING MULTIPOINT SEASON SPLITS

Creating splits at: [0.25, 0.5, 0.75]
Each player-season becomes 3 training samples
  - 25% split: 'At 40 games, predict remaining 122 games WAR'
  - 50% split: 'At 81 games, predict remaining 81 games WAR'
  - 75% split: 'At 121 games, predict remaining 41 games WAR'

Splitting hitter data (from PROCESSED - has Age + WAR_per_600)...

Splitting pitcher data (from PROCESSED - has Age + WAR_per_162)...

Hitter splits: 12702 samples (from 4236 full seasons)
Pitcher splits: 10503 samples (from 5652 full seasons)

Sample split row (hitter):
  Player: David Ortiz
  Year: 2016
  Age: 40
  Split point: 0.25
  Season completion: 25.0%
  Games played: 38
  Current PA: 156
  Remaining PA: 470
  Remaining WAR (target): 3.42

Age preserved in splits:
  Hitters: True (range: 20-45)
  Pitchers: True (range: 19-45)


In [4]:
# Cell 4: Build ROS Features

print("="*70)
print("BUILDING ROS FEATURES")
print("="*70)

# Build complete ROS feature sets (elite detection, age curves, baselines, etc.)
print("\nInitializing feature builders...")
hitter_builder = ROSFeatureBuilder(player_type='hitter')
pitcher_builder = ROSFeatureBuilder(player_type='pitcher')

print("\nBuilding hitter ROS features...")
print("  current_season_df: splits (has Age, WAR_per_600, all stats)")
print("  historical_df: processed (has Age, WAR_per_600 for lookups)")
hitter_with_features = hitter_builder.build_features_batch(
    current_season_df=hitter_splits,  # Has Age + WAR_per_600
    historical_df=hitter_processed  # Has Age + WAR_per_600
)

print("\nBuilding pitcher ROS features...")
pitcher_with_features = pitcher_builder.build_features_batch(
    current_season_df=pitcher_splits,  # Has Age + WAR_per_162
    historical_df=pitcher_processed  # Has Age + WAR_per_162
)

print(f"\nFeature building complete!")
print(f"  Hitter samples with features: {len(hitter_with_features)}")
print(f"  Pitcher samples with features: {len(pitcher_with_features)}")

# Verify feature columns
hitter_feature_count = len([col for col in hitter_with_features.columns if col in ROS_HITTER_FEATURES])
pitcher_feature_count = len([col for col in pitcher_with_features.columns if col in ROS_PITCHER_FEATURES])

print(f"\nFeature availability check:")
print(f"  Hitter features present: {hitter_feature_count}/{len(ROS_HITTER_FEATURES)} ({hitter_feature_count/len(ROS_HITTER_FEATURES)*100:.1f}%)")
print(f"  Pitcher features present: {pitcher_feature_count}/{len(ROS_PITCHER_FEATURES)} ({pitcher_feature_count/len(ROS_PITCHER_FEATURES)*100:.1f}%)")

# Stop if too many features missing
if hitter_feature_count < len(ROS_HITTER_FEATURES) * 0.8:
    raise ValueError(f"Too many hitter features missing: {hitter_feature_count}/{len(ROS_HITTER_FEATURES)}")
if pitcher_feature_count < len(ROS_PITCHER_FEATURES) * 0.8:
    raise ValueError(f"Too many pitcher features missing: {pitcher_feature_count}/{len(ROS_PITCHER_FEATURES)}")

print("\n" + "="*70)

BUILDING ROS FEATURES

Initializing feature builders...

Building hitter ROS features...
  current_season_df: splits (has Age, WAR_per_600, all stats)
  historical_df: processed (has Age, WAR_per_600 for lookups)

Building pitcher ROS features...

Feature building complete!
  Hitter samples with features: 12702
  Pitcher samples with features: 10503

Feature availability check:
  Hitter features present: 65/65 (100.0%)
  Pitcher features present: 51/51 (100.0%)



In [5]:
# Cell 5: Prepare Training Data

print("="*70)
print("PREPARING TRAINING DATA")
print("="*70)

# Use feature-enriched DataFrames from Cell 4
print("\nPreparing hitter training data...")
hitter_train_df, X_hitter, y_hitter = prepare_ros_training_data(
    multipoint_df=hitter_with_features,  # From Cell 4 (has all features)
    feature_columns=ROS_HITTER_FEATURES,
    target_column='remaining_WAR'
)

print("\nPreparing pitcher training data...")
pitcher_train_df, X_pitcher, y_pitcher = prepare_ros_training_data(
    multipoint_df=pitcher_with_features,  # From Cell 4 (has all features)
    feature_columns=ROS_PITCHER_FEATURES,
    target_column='remaining_WAR'
)

print(f"\n{'='*70}")
print("TRAINING DATA SUMMARY")
print(f"{'='*70}")

print(f"\nHitters:")
print(f"  Samples: {len(X_hitter)}")
print(f"  Features: {X_hitter.shape[1]} (expected: {len(ROS_HITTER_FEATURES)})")
print(f"  Target (remaining WAR):")
print(f"    Mean: {y_hitter.mean():.2f}")
print(f"    Std: {y_hitter.std():.2f}")
print(f"    Range: [{y_hitter.min():.2f}, {y_hitter.max():.2f}]")

print(f"\nPitchers:")
print(f"  Samples: {len(X_pitcher)}")
print(f"  Features: {X_pitcher.shape[1]} (expected: {len(ROS_PITCHER_FEATURES)})")
print(f"  Target (remaining WAR):")
print(f"    Mean: {y_pitcher.mean():.2f}")
print(f"    Std: {y_pitcher.std():.2f}")
print(f"    Range: [{y_pitcher.min():.2f}, {y_pitcher.max():.2f}]")

# Verify feature counts match
assert X_hitter.shape[1] == len(ROS_HITTER_FEATURES), f"Hitter feature mismatch: {X_hitter.shape[1]} != {len(ROS_HITTER_FEATURES)}"
assert X_pitcher.shape[1] == len(ROS_PITCHER_FEATURES), f"Pitcher feature mismatch: {X_pitcher.shape[1]} != {len(ROS_PITCHER_FEATURES)}"

print(f"\n{'='*70}")
print("READY FOR TRAINING")
print(f"{'='*70}")

PREPARING TRAINING DATA

Preparing hitter training data...

Preparing pitcher training data...

TRAINING DATA SUMMARY

Hitters:
  Samples: 12702
  Features: 65 (expected: 65)
  Target (remaining WAR):
    Mean: 0.58
    Std: 0.96
    Range: [-1.91, 8.50]

Pitchers:
  Samples: 10503
  Features: 51 (expected: 51)
  Target (remaining WAR):
    Mean: 0.48
    Std: 0.73
    Range: [-1.03, 6.78]

READY FOR TRAINING


In [6]:
# Cell 6: Train Hitter ROS Ensemble

print("="*70)
print("TRAINING HITTER ROS ENSEMBLE")
print("="*70)

# Initialize ensemble with validated weights
print("\nInitializing HitterROSEnsemble...")
hitter_ros = HitterROSEnsemble(
    weights=[0.5, 0.4, 0.1],  # DirectROSForecaster, DartsTemporalEnsemble, Baseline
    feature_columns=ROS_HITTER_FEATURES,
    target_column='remaining_WAR'
)

# Train on multipoint historical data
print("\nFitting ensemble on historical data (2016-2024)...")
print("This may take several minutes...")

hitter_ros.fit(
    historical_df=hitter_train_df,
    feature_columns=ROS_HITTER_FEATURES,
    target_column='remaining_WAR'
)

print("\n" + "="*70)
print("HITTER ROS ENSEMBLE TRAINING COMPLETE")
print("="*70)

TRAINING HITTER ROS ENSEMBLE

Initializing HitterROSEnsemble...

Fitting ensemble on historical data (2016-2024)...
This may take several minutes...
Fitting HitterROSEnsemble on 12702 samples...
  Converting to sktime format (DirectROSForecaster)...
  Fitting DirectROSForecaster (11835 samples after validation)...
  Converting to Darts format (Temporal ensemble)...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


  Fitting DartsTemporalEnsemble (509 player series)...


c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\pytorch_lightning\core\module.py:512: You called `self.log('train_Bias', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\pytorch_lightning\core\module.py:512: You called `self.log('train_Elite_Bias', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\pytorch_lightning\core\module.py:512: You called `self.log('train_Elite_MAE', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\pytorch_lightning\core\module.py:512: You called `self.log('train_MAE', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=A

    Note: AutoARIMA skipped for 509/509 players (<10 years data)
  Preparing baseline training data...
  Fitting baseline MultiQuantileHistGB (12702 samples)...
Ensemble fitting complete.

HITTER ROS ENSEMBLE TRAINING COMPLETE


In [7]:
# Cell 7: Train Pitcher ROS Ensemble

print("="*70)
print("TRAINING PITCHER ROS ENSEMBLE")
print("="*70)

# Initialize ensemble with validated weights
print("\nInitializing PitcherROSEnsemble...")
pitcher_ros = PitcherROSEnsemble(
    weights=[0.5, 0.4, 0.1],  # DirectROSForecaster, DartsTemporalEnsemble, Baseline
    feature_columns=ROS_PITCHER_FEATURES,
    target_column='remaining_WAR'
)

# Train on multipoint historical data
print("\nFitting ensemble on historical data (2016-2024)...")
print("This may take several minutes...")

pitcher_ros.fit(
    historical_df=pitcher_train_df,
    feature_columns=ROS_PITCHER_FEATURES,
    target_column='remaining_WAR'
)

print("\n" + "="*70)
print("PITCHER ROS ENSEMBLE TRAINING COMPLETE")
print("="*70)

TRAINING PITCHER ROS ENSEMBLE

Initializing PitcherROSEnsemble...

Fitting ensemble on historical data (2016-2024)...
This may take several minutes...
Fitting PitcherROSEnsemble on 10503 samples...
  Converting to sktime format (DirectROSForecaster)...
  Fitting DirectROSForecaster (9321 samples after validation)...
  Converting to Darts format (Temporal ensemble)...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


  Fitting DartsTemporalEnsemble (387 player series)...


c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\pytorch_lightning\core\module.py:512: You called `self.log('train_Bias', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\pytorch_lightning\core\module.py:512: You called `self.log('train_Elite_Bias', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\pytorch_lightning\core\module.py:512: You called `self.log('train_Elite_MAE', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\pytorch_lightning\core\module.py:512: You called `self.log('train_MAE', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=A

    Note: AutoARIMA skipped for 387/387 players (<10 years data)
  Preparing baseline training data...
  Fitting baseline MultiQuantileHistGB (10503 samples)...
Ensemble fitting complete.

PITCHER ROS ENSEMBLE TRAINING COMPLETE


In [8]:
# Cell 8: Validate Training

print("="*70)
print("VALIDATING TRAINED MODELS")
print("="*70)

# Test hitter predictions on small sample
print("\nTesting hitter ROS predictions...")
sample_size = 10
# Use DataFrame slicing and pass historical data for cascading fallback
hitter_sample_pred = hitter_ros.predict(
    hitter_train_df.iloc[:sample_size],  # DataFrame with playerid column
    hitter_train_df  # Historical data for tier-based predictions
)
print(f"  Sample predictions: {hitter_sample_pred}")
print(f"  Sample actuals:     {y_hitter[:sample_size]}")
print(f"  Prediction shape: {hitter_sample_pred.shape}")

# Test pitcher predictions
print("\nTesting pitcher ROS predictions...")
pitcher_sample_pred = pitcher_ros.predict(
    pitcher_train_df.iloc[:sample_size],  # DataFrame with playerid column
    pitcher_train_df  # Historical data for tier-based predictions
)
print(f"  Sample predictions: {pitcher_sample_pred}")
print(f"  Sample actuals:     {y_pitcher[:sample_size]}")
print(f"  Prediction shape: {pitcher_sample_pred.shape}")

# Verify predictions are reasonable
assert len(hitter_sample_pred) == sample_size, "Hitter prediction failed"
assert len(pitcher_sample_pred) == sample_size, "Pitcher prediction failed"
assert not np.isnan(hitter_sample_pred).any(), "Hitter predictions contain NaN"
assert not np.isnan(pitcher_sample_pred).any(), "Pitcher predictions contain NaN"

print("\n" + "="*70)
print("VALIDATION PASSED")
print("="*70)

VALIDATING TRAINED MODELS

Testing hitter ROS predictions...
  Player tiers: Tier1=0, Tier2=4, Tier3=6
  Sample predictions: [ 3.25948831  2.31879705  1.13986761 -0.6464828  -0.49127437 -0.30098612
  3.3896042   2.33639773  1.14461728  2.09350332]
  Sample actuals:     [ 3.42404837  2.28269891  1.14134946 -0.91947216 -0.61298144 -0.30649072
  3.86893033  2.57928689  1.28964344  1.9195067 ]
  Prediction shape: (10,)

Testing pitcher ROS predictions...
  Player tiers: Tier1=0, Tier2=9, Tier3=1
  Sample predictions: [ 1.83068192  1.26843484  0.63026315  0.27489989  0.16214536  0.10352663
  0.28874524  0.19055019  0.13108068 -0.00921941]
  Sample actuals:     [ 1.82988485  1.21992323  0.60996162  0.27455395  0.18303597  0.09151798
  0.32393507  0.21595671  0.10797836 -0.00317843]
  Prediction shape: (10,)

VALIDATION PASSED


In [9]:
# Cell 9: Save Trained Models and Historical Split Data

print("="*70)
print("SAVING TRAINED MODELS AND SPLIT DATA")
print("="*70)

# Create models directory
models_dir = project_root / 'models'
models_dir.mkdir(exist_ok=True)

# Save hitter ROS model
hitter_path = models_dir / 'hitter_ros_2025.pkl'
joblib.dump(hitter_ros, hitter_path)
print(f"\nSaved: {hitter_path}")

# Verify file size (trained models should be > 100KB)
hitter_size_mb = hitter_path.stat().st_size / 1024 / 1024
print(f"  File size: {hitter_size_mb:.1f} MB")
assert hitter_size_mb > 0.1, f"Hitter model too small ({hitter_size_mb:.1f} MB) - likely not trained"

# Save hitter Darts temporal models separately (they don't serialize with joblib)
if hitter_ros.temporal_model_fitted:
    print("\nSaving hitter Darts temporal models...")
    hitter_tcn_path = models_dir / 'hitter_ros_tcn_2025.pt'
    hitter_tsmixer_path = models_dir / 'hitter_ros_tsmixer_2025.pt'
    
    hitter_ros.temporal_model.tcn.save(str(hitter_tcn_path))
    hitter_ros.temporal_model.tsmixer.save(str(hitter_tsmixer_path))
    
    print(f"  Saved TCN: {hitter_tcn_path.name}")
    print(f"  Saved TSMixer: {hitter_tsmixer_path.name}")
else:
    print("\n  Note: Hitter temporal model not fitted, skipping Darts model save")

# Save pitcher ROS model
pitcher_path = models_dir / 'pitcher_ros_2025.pkl'
joblib.dump(pitcher_ros, pitcher_path)
print(f"\nSaved: {pitcher_path}")

# Verify file size
pitcher_size_mb = pitcher_path.stat().st_size / 1024 / 1024
print(f"  File size: {pitcher_size_mb:.1f} MB")
assert pitcher_size_mb > 0.1, f"Pitcher model too small ({pitcher_size_mb:.1f} MB) - likely not trained"

# Save pitcher Darts temporal models separately (they don't serialize with joblib)
if pitcher_ros.temporal_model_fitted:
    print("\nSaving pitcher Darts temporal models...")
    pitcher_tcn_path = models_dir / 'pitcher_ros_tcn_2025.pt'
    pitcher_tsmixer_path = models_dir / 'pitcher_ros_tsmixer_2025.pt'
    
    pitcher_ros.temporal_model.tcn.save(str(pitcher_tcn_path))
    pitcher_ros.temporal_model.tsmixer.save(str(pitcher_tsmixer_path))
    
    print(f"  Saved TCN: {pitcher_tcn_path.name}")
    print(f"  Saved TSMixer: {pitcher_tsmixer_path.name}")
else:
    print("\n  Note: Pitcher temporal model not fitted, skipping Darts model save")

print("\n" + "="*70)
print("SAVING HISTORICAL SPLIT DATA (for inference)")
print("="*70)

# Save historical split data (with remaining_WAR) for inference
# This data will be used to provide historical context during ROS predictions
hitter_splits_path = models_dir / 'hitter_splits_2016_2024.pkl'
joblib.dump(hitter_with_features, hitter_splits_path)
print(f"\nSaved: {hitter_splits_path}")

hitter_splits_size_mb = hitter_splits_path.stat().st_size / 1024 / 1024
print(f"  File size: {hitter_splits_size_mb:.1f} MB")
print(f"  Rows: {len(hitter_with_features)}")
print(f"  Columns: {len(hitter_with_features.columns)}")
print(f"  Split points: {sorted(hitter_with_features['split_point'].unique())}")

pitcher_splits_path = models_dir / 'pitcher_splits_2016_2024.pkl'
joblib.dump(pitcher_with_features, pitcher_splits_path)
print(f"\nSaved: {pitcher_splits_path}")

pitcher_splits_size_mb = pitcher_splits_path.stat().st_size / 1024 / 1024
print(f"  File size: {pitcher_splits_size_mb:.1f} MB")
print(f"  Rows: {len(pitcher_with_features)}")
print(f"  Columns: {len(pitcher_with_features.columns)}")
print(f"  Split points: {sorted(pitcher_with_features['split_point'].unique())}")

print("\n" + "="*70)
print("MODELS AND SPLIT DATA SAVED SUCCESSFULLY")
print("="*70)
print("\nSaved files:")
print(f"  - {hitter_path.name} (trained model)")
if hitter_ros.temporal_model_fitted:
    print(f"  - hitter_ros_tcn_2025.pt (Darts TCN model)")
    print(f"  - hitter_ros_tsmixer_2025.pt (Darts TSMixer model)")
print(f"  - {hitter_splits_path.name} (historical splits with remaining_WAR)")
print(f"  - {pitcher_path.name} (trained model)")
if pitcher_ros.temporal_model_fitted:
    print(f"  - pitcher_ros_tcn_2025.pt (Darts TCN model)")
    print(f"  - pitcher_ros_tsmixer_2025.pt (Darts TSMixer model)")
print(f"  - {pitcher_splits_path.name} (historical splits with remaining_WAR)")
print("\nNext: Load these in oWAR_overview.ipynb for inference")

SAVING TRAINED MODELS AND SPLIT DATA

Saved: c:\Users\nairs\Documents\GithubProjects\oWAR\models\hitter_ros_2025.pkl
  File size: 68.6 MB

Saving hitter Darts temporal models...
  Saved TCN: hitter_ros_tcn_2025.pt
  Saved TSMixer: hitter_ros_tsmixer_2025.pt

Saved: c:\Users\nairs\Documents\GithubProjects\oWAR\models\pitcher_ros_2025.pkl
  File size: 50.0 MB

Saving pitcher Darts temporal models...
  Saved TCN: pitcher_ros_tcn_2025.pt
  Saved TSMixer: pitcher_ros_tsmixer_2025.pt

SAVING HISTORICAL SPLIT DATA (for inference)

Saved: c:\Users\nairs\Documents\GithubProjects\oWAR\models\hitter_splits_2016_2024.pkl
  File size: 13.5 MB
  Rows: 12702
  Columns: 140
  Split points: [np.float64(0.25), np.float64(0.5), np.float64(0.75)]

Saved: c:\Users\nairs\Documents\GithubProjects\oWAR\models\pitcher_splits_2016_2024.pkl
  File size: 9.5 MB
  Rows: 10503
  Columns: 119
  Split points: [np.float64(0.25), np.float64(0.5), np.float64(0.75)]

MODELS AND SPLIT DATA SAVED SUCCESSFULLY

Saved files:

In [21]:
# Cell 9.5: Pitcher ROS WAR Diagnostic (STARTERS + RELIEVERS)

# Suppress PyTorch Lightning verbosity for cleaner output
import os
import warnings
import logging

os.environ['PYTORCH_LIGHTNING_VERBOSITY'] = '0'
warnings.filterwarnings('ignore', category=UserWarning, module='pytorch_lightning')
warnings.filterwarnings('ignore', category=FutureWarning, module='pytorch_lightning')
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning.utilities.rank_zero").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning.accelerators.cuda").setLevel(logging.ERROR)

print("="*90)
print("PITCHER ROS WAR DIAGNOSTIC (STARTERS + RELIEVERS)")
print("="*90)
print()

# Load and process current season data
from new_pipeline.notebooks.shared.pipeline_runner import load_current_season_data, run_data_pipeline

# Reload modules to ensure we have the latest version
import importlib
import new_pipeline.common.projections.usage_projections
import new_pipeline.common.projections.ros_projections
importlib.reload(new_pipeline.common.projections.usage_projections)
importlib.reload(new_pipeline.common.projections.ros_projections)

from new_pipeline.common.projections.usage_projections import (
    get_team_games_from_data,
    calculate_pitcher_remaining_ip
)
from new_pipeline.common.projections.ros_projections import format_pitcher_ros_display

pitcher_2025_raw = load_current_season_data('pitcher', year=2025)
pitcher_2025_processed = run_data_pipeline(pitcher_2025_raw, player_type='pitcher')

# Load hitter data to get accurate team games played
hitter_2025_raw = load_current_season_data('hitter', year=2025)
hitter_2025_processed = run_data_pipeline(hitter_2025_raw, player_type='hitter')

# Get team-specific games played using utility function
team_games_dict, league_median_games = get_team_games_from_data(hitter_2025_processed)

# For display summary
season_pct = league_median_games / 162
remaining_team_games = 162 - league_median_games

print(f"League median games played: {league_median_games:.0f}, Remaining: {remaining_team_games:.0f}, Season: {season_pct:.1%}")
print(f"(Note: Individual projections use team-specific games + multi-team handling)")
print()

# ============================================================================
# STARTERS (GS/G > 0.7)
# ============================================================================
print("="*90)
print("STARTERS (Top 20 by IP)")
print("="*90)

starter_mask = (pitcher_2025_processed['GS'] / pitcher_2025_processed['G']) > 0.7
starters = pitcher_2025_processed[starter_mask].nlargest(20, 'IP')

print(f"Building ROS features for {len(starters)} top starters...")

# Build ROS features
pitcher_ros_features_starters = pitcher_builder.build_features_batch(
    current_season_df=starters,
    historical_df=pitcher_processed,
    injury_df=None
)

# Get ROS predictions with uncertainty
ros_predictions_starters = pitcher_ros.predict_with_uncertainty(
    current_df=pitcher_ros_features_starters,
    historical_df=pitcher_with_features
)

# Calculate tier classification using baseline model's q50
X_starters = pitcher_ros_features_starters[ROS_PITCHER_FEATURES].values
baseline_quantiles_starters = pitcher_ros.baseline_model.predict_quantiles(X_starters)
baseline_q50_starters = baseline_quantiles_starters[0.5]

# Calculate dynamic thresholds for starters (WAR_per_162 rates)
base_good = 3.3
base_elite = 5.0
scaling = min(1.22 * season_pct**2 - 0.83 * season_pct + 0.61, 1.0)
good_threshold_starters = base_good * scaling
elite_threshold_starters = base_elite * scaling

# Classify tiers based on baseline q50
tier_labels_starters = np.array([
    'average' if w < good_threshold_starters 
    else 'good' if w < elite_threshold_starters 
    else 'elite' 
    for w in baseline_q50_starters
])

# Calculate remaining IP using utility function
remaining_ip_results = []
for _, starter in starters.iterrows():
    result = calculate_pitcher_remaining_ip(
        current_ip=starter['IP'],
        pitcher_games=starter['G'],
        team=starter['Team'],
        team_games_dict=team_games_dict,
        league_median_games=league_median_games,
        games_started=starter['GS']
    )
    remaining_ip_results.append(result)

# Extract results
projected_remaining_ip = np.array([r['remaining_ip'] for r in remaining_ip_results])
starter_team_games = np.array([r['team_games_played'] for r in remaining_ip_results])
ip_per_start = np.array([r['ip_per_appearance'] for r in remaining_ip_results])

# Format ROS predictions for display using utility function
# IMPORTANT: ROS model predicts cumulative WAR directly, not rates!
ros_display_starters = format_pitcher_ros_display(
    ros_predictions_starters,
    projected_remaining_ip,
    role='starter',
    include_rates=True
)

# Display starter diagnostic table
diag_df_starters = pd.DataFrame({
    'Name': starters['Name'].values,
    'Team': starters['Team'].values,
    'Tier': tier_labels_starters,
    'IP': np.round(starters['IP'].values, 0),
    'Starts': np.round(starters['GS'].values, 0),
    'IP/Start': np.round(ip_per_start, 1),
    'TeamG': starter_team_games,
    'Proj_IP': np.round(projected_remaining_ip, 0),
    'ROS_Rate': np.round(ros_display_starters['ros_rate'], 2),
    'ROS_WAR': np.round(ros_display_starters['ros_war'], 1),
    'ROS_q50': np.round(ros_display_starters['ros_q50'], 1),
    'ROS_q90': np.round(ros_display_starters['ros_q90'], 1)
})

print(diag_df_starters.to_string(index=False))
print()

# Tier distribution summary
elite_count_starters = (tier_labels_starters == 'elite').sum()
good_count_starters = (tier_labels_starters == 'good').sum()
avg_count_starters = (tier_labels_starters == 'average').sum()

print(f"Tier Distribution: Elite={elite_count_starters}, Good={good_count_starters}, Average={avg_count_starters}")
print(f"Thresholds (WAR_per_162 rates): good={good_threshold_starters:.2f}, elite={elite_threshold_starters:.2f}")
print()

# ============================================================================
# RELIEVERS (GS/G <= 0.7)
# ============================================================================
print("="*90)
print("RELIEVERS (Top 20 by Games Pitched)")
print("="*90)

reliever_mask = (pitcher_2025_processed['GS'] / pitcher_2025_processed['G']) <= 0.7
relievers = pitcher_2025_processed[reliever_mask].nlargest(20, 'G')

print(f"Building ROS features for {len(relievers)} top relievers...")

# Build ROS features
pitcher_ros_features_relievers = pitcher_builder.build_features_batch(
    current_season_df=relievers,
    historical_df=pitcher_processed,
    injury_df=None
)

# Get ROS predictions with uncertainty
ros_predictions_relievers = pitcher_ros.predict_with_uncertainty(
    current_df=pitcher_ros_features_relievers,
    historical_df=pitcher_with_features
)

# Calculate tier classification using baseline model's q50
X_relievers = pitcher_ros_features_relievers[ROS_PITCHER_FEATURES].values
baseline_quantiles_relievers = pitcher_ros.baseline_model.predict_quantiles(X_relievers)
baseline_q50_relievers = baseline_quantiles_relievers[0.5]

# Calculate dynamic thresholds for relievers (WAR_per_48.2 rates)
base_good_full = 1.5
base_elite_full = 2.25
typical_ip = 70
base_good = base_good_full / typical_ip * 48.2   # = 1.03
base_elite = base_elite_full / typical_ip * 48.2  # = 1.55
scaling = min(season_pct ** 0.7, 1.0)
good_threshold_relievers = base_good * scaling
elite_threshold_relievers = base_elite * scaling

# Classify tiers based on baseline q50
tier_labels_relievers = np.array([
    'average' if w < good_threshold_relievers 
    else 'good' if w < elite_threshold_relievers 
    else 'elite' 
    for w in baseline_q50_relievers
])

# Calculate remaining IP using utility function
remaining_ip_results_rel = []
for _, reliever in relievers.iterrows():
    result = calculate_pitcher_remaining_ip(
        current_ip=reliever['IP'],
        pitcher_games=reliever['G'],
        team=reliever['Team'],
        team_games_dict=team_games_dict,
        league_median_games=league_median_games,
        games_started=reliever['GS']
    )
    remaining_ip_results_rel.append(result)

# Extract results
projected_remaining_ip_rel = np.array([r['remaining_ip'] for r in remaining_ip_results_rel])
reliever_team_games = np.array([r['team_games_played'] for r in remaining_ip_results_rel])
ip_per_appearance = np.array([r['ip_per_appearance'] for r in remaining_ip_results_rel])
appearance_rate = np.array([r['appearance_rate'] for r in remaining_ip_results_rel])

# Format ROS predictions for display using utility function
# IMPORTANT: ROS model predicts cumulative WAR directly, not rates!
ros_display_relievers = format_pitcher_ros_display(
    ros_predictions_relievers,
    projected_remaining_ip_rel,
    role='reliever',
    include_rates=True
)

# Display reliever diagnostic table
diag_df_relievers = pd.DataFrame({
    'Name': relievers['Name'].values,
    'Team': relievers['Team'].values,
    'Tier': tier_labels_relievers,
    'IP': np.round(relievers['IP'].values, 0),
    'G': np.round(relievers['G'].values, 0),
    'IP/G': np.round(ip_per_appearance, 2),
    'App_Rate': np.round(appearance_rate, 2),
    'TeamG': reliever_team_games,
    'Proj_IP': np.round(projected_remaining_ip_rel, 0),
    'ROS_Rate': np.round(ros_display_relievers['ros_rate'], 2),
    'ROS_WAR': np.round(ros_display_relievers['ros_war'], 1),
    'ROS_q50': np.round(ros_display_relievers['ros_q50'], 1),
    'ROS_q90': np.round(ros_display_relievers['ros_q90'], 1)
})

print(diag_df_relievers.to_string(index=False))
print()

# Tier distribution summary
elite_count_relievers = (tier_labels_relievers == 'elite').sum()
good_count_relievers = (tier_labels_relievers == 'good').sum()
avg_count_relievers = (tier_labels_relievers == 'average').sum()

print(f"Tier Distribution: Elite={elite_count_relievers}, Good={good_count_relievers}, Average={avg_count_relievers}")
print(f"Thresholds (WAR_per_48.2 rates): good={good_threshold_relievers:.2f}, elite={elite_threshold_relievers:.2f}")
print()

# ============================================================================
# COLUMN GUIDES
# ============================================================================
print("="*90)
print("COLUMN GUIDE")
print("="*90)
print()
print("STARTERS:")
print("  Team = Player's team (or '- - -' for multi-team players)")
print("  Tier = Classification (average/good/elite) based on baseline_model q50 vs dynamic thresholds")
print("  IP = Innings pitched so far")
print("  Starts = Games started so far")
print("  IP/Start = Average innings per start")
print("  TeamG = Team games played (or player's G for multi-team)")
print("  Proj_IP = Projected remaining IP (uses usage_projections.calculate_pitcher_remaining_ip)")
print()
print("RELIEVERS:")
print("  Team = Player's team (or '- - -' for multi-team players)")
print("  Tier = Classification (average/good/elite) based on baseline_model q50 vs dynamic thresholds")
print("  IP = Innings pitched so far")
print("  G = Games pitched (appearances)")
print("  IP/G = Average innings per appearance")
print("  App_Rate = Appearance rate (G / TeamG)")
print("  TeamG = Team games played (or player's G for multi-team)")
print("  Proj_IP = Projected remaining IP (uses usage_projections.calculate_pitcher_remaining_ip)")
print()
print("COMMON:")
print("  ROS_Rate = Implied WAR rate (WAR_per_162 or WAR_per_48.2) for comparison")
print("  ROS_WAR = Projected rest-of-season cumulative WAR (model prediction)")
print("  ROS_q50, ROS_q90 = Quantile predictions (uncertainty bands)")
print()
print("IMPORTANT NOTE:")
print("  - ROS model predicts CUMULATIVE remaining WAR directly (not rates!)")
print("  - ROS_WAR is the actual prediction to use for projections")
print("  - ROS_Rate is calculated for comparison only: cumulative_war / (remaining_ip / normalization_basis)")
print()
print("Tier Classification:")
print("  - Uses baseline_model.predict_quantiles()[0.5] as conservative anchor")
print("  - Starters: base_good=3.3, base_elite=5.0 (WAR_per_162), quadratic scaling")
print("  - Relievers: base_good=1.03, base_elite=1.55 (WAR_per_48.2), square root scaling")
print()
print("Team-Specific Handling:")
print("  - Uses usage_projections.get_team_games_from_data() and get_team_games_for_player()")
print("  - Single-team players: Use their team's max games played")
print("  - Multi-team players ('- - -'): Use their actual games played")
print("  - Rotation detection and appearance rates calculated by utility functions")
print()
print("Note: Caps: Starters=210 IP, Relievers=100 IP (2025 max observed=90 IP)")
print("="*90)

23:07:01 - new_pipeline.common.transformers.filters - INFO - IPFilter: Removed 157 pitchers (position players / insufficient sample, partial season)
23:07:01 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Loaded Age for 873 pitchers
23:07:01 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Added Age column (range: 20-42)
23:07:01 - new_pipeline.common.transformers.pitcher_features - INFO - Loading pitcher features...


PITCHER ROS WAR DIAGNOSTIC (STARTERS + RELIEVERS)

Loading partial season data: fangraphs_pitchers_2025_firsthalf.csv


23:07:01 - new_pipeline.common.transformers.pitcher_features - INFO - Loaded 13 pitcher feature sets (38 total columns)
23:07:01 - new_pipeline.common.transformers.pitcher_composite_transformer - INFO - Calculating pitcher composite features...
23:07:01 - new_pipeline.common.transformers.pitcher_composite_transformer - INFO - Added 7 composite features
23:07:01 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Learned replacement values for 41 features
23:07:01 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Imputed 883 missing values
23:07:01 - new_pipeline.common.transformers.validators - WARNING - FeatureValidator found issues:
  - Feature 'BB%' range [0.00, 26.92] outside expected [0, 25]
  - Feature 'ERA' range [0.00, 19.86] outside expected [0, 15]
  - Feature 'GB%' range [11.11, 74.71] outside expected [20, 80]
23:07:01 - new_pipeline.common.transformers.feature_selector - INFO - FeatureSelector: Selected 14 features + 10 met

Loading partial season data: fangraphs_hitters_2025_firsthalf.csv


23:07:02 - new_pipeline.common.transformers.hitter_features - INFO - Loaded 11 hitter feature sets (33 total columns)
23:07:02 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Learned replacement values for 28 features
23:07:02 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Imputed 99 missing values
23:07:02 - new_pipeline.common.transformers.validators - WARNING - FeatureValidator found issues:
  - Feature 'AVG' range [0.07, 0.36] outside expected [0.1, 0.4]
  - Feature 'OBP' range [0.12, 0.47] outside expected [0.2, 0.5]
  - Feature 'SLG' range [0.07, 0.74] outside expected [0.2, 0.8]
23:07:02 - new_pipeline.common.transformers.feature_selector - INFO - FeatureSelector: Selected 9 features + 10 metadata columns
23:07:02 - new_pipeline.common.transformers.normalizers - INFO - WARNormalizer: Added 'WAR_per_600' column


League median games played: 95, Remaining: 67, Season: 58.6%
(Note: Individual projections use team-specific games + multi-team handling)

STARTERS (Top 20 by IP)
Building ROS features for 20 top starters...
  Player tiers: Tier1=12, Tier2=1, Tier3=7
              Name Team    Tier    IP  Starts  IP/Start  TeamG  Proj_IP  ROS_Rate  ROS_WAR  ROS_q50  ROS_q90
   Garrett Crochet  BOS    good 129.0      20       6.5     97     84.0      4.54      2.4      2.4      2.4
        Logan Webb  SFG average 125.0      20       6.3     96     83.0      3.26      1.7      1.7      1.3
      Zack Wheeler  PHI    good 122.0      19       6.4     96     85.0      3.74      2.0      2.0      2.2
         Max Fried  NYY average 122.0      20       6.1     96     81.0      2.87      1.4      1.4      1.7
      Tarik Skubal  DET    good 121.0      19       6.4     95     85.0      4.90      2.6      2.6      2.8
       Paul Skenes  PIT    good 121.0      20       6.0     91     86.0      4.35      2.3     

In [22]:
# Cell 9.6: Hitter ROS WAR Diagnostic

# Suppress PyTorch Lightning verbosity for cleaner output
import os
import warnings
import logging

os.environ['PYTORCH_LIGHTNING_VERBOSITY'] = '0'
warnings.filterwarnings('ignore', category=UserWarning, module='pytorch_lightning')
warnings.filterwarnings('ignore', category=FutureWarning, module='pytorch_lightning')
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning.utilities.rank_zero").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning.accelerators.cuda").setLevel(logging.ERROR)

print("="*90)
print("HITTER ROS WAR DIAGNOSTIC")
print("="*90)
print()

# Load and process current season data
# Reload modules to ensure we have the latest version
import importlib
import new_pipeline.common.projections.usage_projections
import new_pipeline.common.projections.ros_projections
importlib.reload(new_pipeline.common.projections.usage_projections)
importlib.reload(new_pipeline.common.projections.ros_projections)

from new_pipeline.common.projections.usage_projections import (
    get_team_games_from_data,
    calculate_hitter_remaining_pa
)
from new_pipeline.common.projections.ros_projections import format_hitter_ros_display

hitter_2025_raw = load_current_season_data('hitter', year=2025)
hitter_2025_processed = run_data_pipeline(hitter_2025_raw, player_type='hitter')

# Get team-specific games played using utility function
team_games_dict, league_median_games = get_team_games_from_data(hitter_2025_processed)

# Get top 20 hitters by PA (to match pipeline diagnostic)
top_20_hitters = hitter_2025_processed.nlargest(20, 'PA')

print(f"Building ROS features for {len(top_20_hitters)} top hitters...")

# Build ROS features
hitter_ros_features = hitter_builder.build_features_batch(
    current_season_df=top_20_hitters,
    historical_df=hitter_processed,
    injury_df=None
)

# Get ROS predictions with uncertainty
ros_predictions = hitter_ros.predict_with_uncertainty(
    current_df=hitter_ros_features,
    historical_df=hitter_with_features
)

# Calculate tier classification using baseline model's q50
X_hitters = hitter_ros_features[ROS_HITTER_FEATURES].values
baseline_quantiles_hitters = hitter_ros.baseline_model.predict_quantiles(X_hitters)
baseline_q50_hitters = baseline_quantiles_hitters[0.5]

# Calculate season progression (for display and tier thresholds)
season_pct = league_median_games / 162
games_remaining = 162 - league_median_games

print(f"League median games played: {league_median_games:.0f}, Remaining: {games_remaining:.0f}, Season: {season_pct:.1%}")
print(f"(Note: Individual projections use team-specific games + multi-team handling)")
print()

# Calculate dynamic thresholds for hitters (WAR_per_600 rates)
base_good = 3.3
base_elite = 5.0
scaling = min(season_pct, 1.0)
good_threshold_hitters = base_good * scaling
elite_threshold_hitters = base_elite * scaling

# Classify tiers based on baseline q50
tier_labels_hitters = np.array([
    'average' if w < good_threshold_hitters 
    else 'good' if w < elite_threshold_hitters 
    else 'elite' 
    for w in baseline_q50_hitters
])

# Calculate remaining PA using utility function
remaining_pa_results = []
for _, hitter in top_20_hitters.iterrows():
    result = calculate_hitter_remaining_pa(
        current_pa=hitter['PA'],
        current_games=hitter['G'],
        team=hitter['Team'],
        team_games_dict=team_games_dict,
        league_median_games=league_median_games
    )
    remaining_pa_results.append(result)

# Extract results
projected_remaining_pa = np.array([r['remaining_pa'] for r in remaining_pa_results])
hitter_team_games = np.array([r['team_games_played'] for r in remaining_pa_results])
pa_per_game = np.array([r['pa_per_game'] for r in remaining_pa_results])
participation_rate = np.array([r['participation_rate'] for r in remaining_pa_results])

# Get current PA for display
current_pa = top_20_hitters['PA'].values

# Format ROS predictions for display using utility function
# IMPORTANT: ROS model predicts cumulative WAR directly, not rates!
ros_display = format_hitter_ros_display(
    ros_predictions,
    projected_remaining_pa,
    include_rates=True
)

# Display diagnostic table
diag_df = pd.DataFrame({
    'Name': top_20_hitters['Name'].values,
    'Team': top_20_hitters['Team'].values,
    'Tier': tier_labels_hitters,
    'PA': np.round(current_pa, 0),
    'G': np.round(top_20_hitters['G'].values, 0),
    'TeamG': hitter_team_games,
    'Remaining_PA': np.round(projected_remaining_pa, 0),
    'Total_PA_Proj': np.round(current_pa + projected_remaining_pa, 0),
    'ROS_Rate': np.round(ros_display['ros_rate'], 2),
    'ROS_WAR': np.round(ros_display['ros_war'], 1),
    'ROS_q50': np.round(ros_display['ros_q50'], 1),
    'ROS_q90': np.round(ros_display['ros_q90'], 1)
})

print(diag_df.to_string(index=False))
print()

# Tier distribution summary
elite_count_hitters = (tier_labels_hitters == 'elite').sum()
good_count_hitters = (tier_labels_hitters == 'good').sum()
avg_count_hitters = (tier_labels_hitters == 'average').sum()

print(f"Tier Distribution: Elite={elite_count_hitters}, Good={good_count_hitters}, Average={avg_count_hitters}")
print(f"Thresholds (WAR_per_600 rates): good={good_threshold_hitters:.2f}, elite={elite_threshold_hitters:.2f}")
print()

print("="*90)
print("Column Guide:")
print("  Team = Player's team (or '- - -' for multi-team players)")
print("  Tier = Classification (average/good/elite) based on baseline_model q50 vs dynamic thresholds")
print("  PA = Plate appearances so far")
print("  G = Games played so far")
print("  TeamG = Team games played (or player's G for multi-team)")
print("  Remaining_PA = Projected remaining PA (uses usage_projections.calculate_hitter_remaining_pa)")
print("  Total_PA_Proj = Total projected season PAs (current + remaining)")
print("  ROS_Rate = Implied WAR_per_600 for comparison")
print("  ROS_WAR = Projected rest-of-season cumulative WAR (model prediction)")
print("  ROS_q50, ROS_q90 = Quantile predictions (uncertainty bands)")
print()
print("IMPORTANT NOTE:")
print("  - ROS model predicts CUMULATIVE remaining WAR directly (not rates!)")
print("  - ROS_WAR is the actual prediction to use for projections")
print("  - ROS_Rate is calculated for comparison only: cumulative_war / (remaining_pa / 600)")
print()
print("Tier Classification:")
print("  - Uses baseline_model.predict_quantiles()[0.5] as conservative anchor")
print("  - base_good=3.3, base_elite=5.0 (WAR_per_600), linear scaling")
print()
print("Caps Applied (by utility function):")
print("  1. Season total cap: 780 PAs (leadoff everyday ~762, +buffer)")
print("  2. PA/game rate cap: 5.0 PA/G (normal 3.8-4.7 by lineup spot)")
print("  Note: Preserves natural PA distribution based on observed rates")
print()
print("Team-Specific Handling:")
print("  - Uses usage_projections.get_team_games_from_data() and get_team_games_for_player()")
print("  - Single-team players: Use their team's max games played")
print("  - Multi-team players ('- - -'): Use their actual games played")
print("  - Participation rate and remaining PA projections calculated by utility functions")
print()
print("Compare Total_PA_Proj - should range 620-780 PAs based on lineup position")
print("="*90)

23:07:07 - new_pipeline.common.transformers.filters - INFO - PAFilter: Removed 118 hitters with < 37 PA (partial season)
23:07:07 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Loaded Age for 673 hitters
23:07:07 - new_pipeline.common.transformers.age_enricher - INFO - AgeEnricher: Added Age column (range: 21-41)
23:07:07 - new_pipeline.common.transformers.hitter_features - INFO - Loading hitter features...


HITTER ROS WAR DIAGNOSTIC

Loading partial season data: fangraphs_hitters_2025_firsthalf.csv


23:07:08 - new_pipeline.common.transformers.hitter_features - INFO - Loaded 11 hitter feature sets (33 total columns)
23:07:08 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Learned replacement values for 28 features
23:07:08 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Imputed 99 missing values
23:07:08 - new_pipeline.common.transformers.validators - WARNING - FeatureValidator found issues:
  - Feature 'AVG' range [0.07, 0.36] outside expected [0.1, 0.4]
  - Feature 'OBP' range [0.12, 0.47] outside expected [0.2, 0.5]
  - Feature 'SLG' range [0.07, 0.74] outside expected [0.2, 0.8]
23:07:08 - new_pipeline.common.transformers.feature_selector - INFO - FeatureSelector: Selected 9 features + 10 metadata columns
23:07:08 - new_pipeline.common.transformers.normalizers - INFO - WARNormalizer: Added 'WAR_per_600' column


Building ROS features for 20 top hitters...
  Player tiers: Tier1=13, Tier2=4, Tier3=3
League median games played: 95, Remaining: 67, Season: 58.6%
(Note: Individual projections use team-specific games + multi-team handling)

            Name  Team    Tier  PA  G  TeamG  Remaining_PA  Total_PA_Proj  ROS_Rate  ROS_WAR  ROS_q50  ROS_q90
   Rafael Devers - - - average 443 98     98         289.0          732.0      2.10      1.0      1.0      1.4
   Shohei Ohtani   LAD    good 438 95     95         309.0          747.0      4.79      2.5      2.5      3.3
    Jarren Duran   BOS average 438 97     97         294.0          732.0      1.80      0.9      0.9      1.1
 Julio Rodríguez   SEA average 431 95     95         304.0          735.0      3.76      1.9      1.9      1.9
Francisco Lindor   NYM average 430 95     97         288.0          718.0      3.74      1.8      1.8      2.8
     Aaron Judge   NYY   elite 429 96     96         295.0          724.0      7.17      3.5      3.5      4